# Data-Science Notebook on SPUR — DuckDB engine
## Design proposal (brainstorm in progress)

> **Date:** 2026-05-29 · **Status:** brainstorming — not yet approved. This notebook is the *visual companion* for the design dialogue (dogfooded through the Jute observe-plane via the notebook MCP).
> **Vision:** Attach a dataset/datasource to a Jute notebook (drag-drop into a sidebar), then drive analysis + visualization by asking the SPUR brain agent from the spur-tui terminal. **DuckDB is the data-processing engine.**

### Decisions locked so far

| # | Decision | Choice |
|---|---|---|
| Scope | datasource kinds | **A** local files (CSV/Parquet/JSON) → **B** DuckDB/SQLite (incl. spur-analyst index) → **C/D** remote DB / object-store *later* |
| Engine | data processing | **DuckDB** (the engine; not pandas) |
| Execution | where DuckDB runs | **A** inside the Python kernel (`duckdb` package) — reuses the existing kernel + viz render path |
| Attach UX | entrypoint | **GUI sidebar** — drag-drop a file or “Add datasource” button |

### Still open (your call — see §6)
- How an attached datasource is **represented** (setup cell vs daemon catalog vs both).
- How the **brain becomes schema-aware**.

## 1. Why DuckDB — the extension ecosystem makes A→B→C/D one arc, not three rewrites

The key insight: choosing DuckDB means the scope expansion is mostly **“enable an extension,”** not “build a connector.” The same SQL surface that reads a local CSV today attaches Postgres or S3 later.

| Scope | Data reached | DuckDB mechanism | New plumbing |
|---|---|---|---|
| **A** (MVP) | CSV / Parquet / JSON (local) | `read_csv_auto` / `read_parquet` / `read_json_auto` (built-in) | none |
| **B** | DuckDB / SQLite files, **spur-analyst graph index** | `ATTACH`, `sqlite_scanner` | minimal |
| **C** (later) | Postgres / MySQL | `postgres_scanner` / `mysql_scanner` + connection string | secrets mgmt |
| **D** (later) | S3 / https / Iceberg / Delta | `httpfs` / `iceberg` / `delta` | credentials |

**Precedent already in the tree:** `spur-analyst` runs DuckDB as its query substrate (queried via `mcp__spur-analyst__query`), so the dependency and the “DuckDB-as-substrate” pattern already exist. And **visualization already works** — the kernel emits `display_data`/`execute_result`; Jute renders `image/png` via `OutputView.tsx` (`df.plot()` / plotly render today). So the genuinely new surface is just **attach + brain-awareness**, not the run/render path.

## 2. System architecture — one store, three transports, now carrying a datasource

This extends the existing daemon architecture (companion: `2026-05-28-spur-notebook-jute-notebook-architecture.ipynb`). The new piece is a typed **`AttachDatasource`** command on the existing control plane, plus a DuckDB-in-kernel runtime.

```mermaid
flowchart TB
  subgraph TUI["spur-tui (terminal)"]
    U1["user: '/analyze sales by region'"]
    BRAIN["brain agent (ACP)"]
    U1 --> BRAIN
  end
  subgraph GUI["Jute GUI (the daemon window)"]
    SB["NEW: Datasource sidebar<br/>drag-drop / Add datasource"]
    CELLS["notebook cells + outputs (charts)"]
  end
  subgraph DAEMON["jute-notebook daemon (single store)"]
    CTL["DaemonControlCommand router<br/>+ NEW AttachDatasource"]
    STORE["authoritative NotebookStore"]
    CAT["NEW: datasource catalog (schema)"]
  end
  subgraph KERNEL["Python kernel"]
    DUCK["duckdb connection<br/>read_csv_auto / CREATE VIEW"]
  end
  SB -- AttachDatasource --> CTL
  BRAIN -- notebook_* MCP (insert/run cell) --> CTL
  CTL --> STORE
  CTL --> CAT
  STORE -- setup cell + analysis cells --> KERNEL
  KERNEL --> DUCK
  DUCK -- Arrow/df --> CELLS
  STORE -- DeltaKind --> CELLS
```

**The seam to respect:** the *attach* happens in the **GUI sidebar**, but the *analysis* is driven by the **brain from the TUI**. This is exactly the TUI↔GUI cross-surface boundary the user-journey audit flagged (§6.2: the two never co-change). The design must make the attach produce an artifact the brain can see — that is what §3/§6 is about.

## 3. End-to-end user journey

```mermaid
sequenceDiagram
  autonumber
  participant U as User
  participant SB as Jute sidebar (GUI)
  participant D as daemon (+catalog)
  participant K as Python kernel (duckdb)
  participant B as Brain (from spur-tui)
  participant C as Notebook cells

  U->>SB: drag sales.csv into sidebar
  SB->>D: AttachDatasource{path, name:'sales'}
  D->>K: run setup cell (CREATE VIEW sales AS read_csv_auto('sales.csv'))
  K-->>D: schema (DESCRIBE sales)
  D->>C: insert reproducible setup cell
  D-->>SB: sidebar shows 'sales' + columns
  Note over U,B: user switches to spur-tui
  U->>B: 'plot monthly revenue by region'
  B->>D: reads setup cell + catalog schema (aware of 'sales')
  B->>D: notebook_insert_cell (duckdb.sql(...).df().plot())
  B->>D: notebook_run_cell
  D->>K: execute
  K-->>C: display_data image/png (chart)
  Note over U,C: user sees the chart render live in Jute
```

The brain never guesses the schema: it either reads the generated setup cell or the catalog (§6).

## 4. The datasource sidebar (GUI) — proposed layout

A new left panel in the notebook view. Two ways in: **drag-drop a file** onto the drop zone, or **“+ Add datasource”** (file picker; later: connection string for C).

```text
┌─ DATA ───────────┐┌─ notebook cells ────────────────┐
│ + Add datasource ││ [1] con.sql("CREATE VIEW sales …")│
│                  ││     (auto-generated setup cell)  │
│ ◈ sales  (csv)   ││                                  │
│   12,840 rows    ││ [2] # brain: monthly revenue…   │
│   region,month,  ││     duckdb.sql(...).df().plot()  │
│   revenue, …     ││     ┌────── chart ─────┐      │
│                  ││     │   ▃ ▇ ▅ ▆ ▄      │      │
│ ◈ graph (duckdb) ││     └───────────────┘      │
│   [drop file…]   ││                                  │
└────────────────┘└─────────────────────────┘
```

Per-source row: **name** (editable alias), **kind** badge (csv/parquet/duckdb), **row count**, **column peek** (from `DESCRIBE`). Click a source → inserts a `SELECT * … LIMIT 100` preview cell. Drag a source name into a cell → inserts its view name. Mirrors the existing recents/cards interaction language in `HomePage.tsx`.

*(This is a first sketch — layout is open for your feedback.)*

## 5. Three approaches to representation + brain-awareness

The one real design fork: when the sidebar attaches a datasource, **what artifact is produced**, and **how does the TUI-driven brain learn the schema?**

```mermaid
flowchart LR
  subgraph A["A — setup cell only"]
    A1["sidebar inserts a code cell"] --> A2["brain reads the cell + runs DESCRIBE"]
  end
  subgraph B["B — daemon catalog only"]
    B1["sidebar -> AttachDatasource"] --> B2["daemon stores schema"] --> B3["schema injected into brain context"]
  end
  subgraph C["C — both (recommended)"]
    C1["AttachDatasource"] --> C2["daemon inserts reproducible setup cell"]
    C1 --> C3["daemon tracks lightweight catalog"]
    C3 --> C4["brain sees schema w/o a round-trip"]
  end
```

| | A: setup cell only | B: catalog only | **C: both (rec.)** |
|---|---|---|---|
| Reproducible (re-run notebook) | ✅ | ❌ (needs daemon) | ✅ |
| Brain schema-aware w/o round-trip | ❌ (needs DESCRIBE) | ✅ | ✅ |
| New persistence / wire types | none | catalog + event | catalog + 1 command |
| Headless (§10-④ of audit) | ❌ | ✅ | ✅ |
| MVP cost | lowest | medium | low–medium |

**Recommendation: C, lightweight.** The sidebar emits one typed `AttachDatasource` control command (same plane as `Open`/`New`). The daemon then does two cheap things: (1) **inserts a reproducible setup cell** so the notebook still works standalone, and (2) **records a small catalog entry** (name, path, kind, columns) so the brain — driven from the TUI — is schema-aware **without** a DESCRIBE round-trip. This is the option that actually closes the TUI↔GUI seam: the GUI attach becomes immediately legible to the TUI brain.

## 6. Open questions for you

1. **Representation (§5):** go with **C (setup cell + lightweight catalog)**, or keep MVP dead-simple with **A (setup cell only, brain runs `DESCRIBE`)**?
2. **Brain awareness:** if C — inject the catalog schema into the brain's **session context** (always-on, costs prompt tokens) or expose a **`notebook_list_datasources` MCP tool** the brain calls on demand (lazy, one round-trip)?
3. **Attach entrypoint:** GUI sidebar only for MVP, or *also* a TUI `/notebook data add <path>` so a terminal-first user never leaves the keyboard?
4. **MVP demo / success criterion:** is the bar “drag a CSV → ask brain in TUI → chart renders in Jute”? Any specific dataset you want as the canonical demo (e.g. the spur-analyst graph index, exercising scope B)?
5. **Sidebar layout (§4):** does the left-panel sketch fit, or do you picture it elsewhere (right panel / collapsible / a `DATA` tab)?

### Proposed YAGNI cut-lines for the MVP
- Scope **A only** first (local CSV/Parquet/JSON); B (DuckDB/SQLite + spur-analyst) as fast-follow; C/D deferred to extension-enable.
- No credential/secret management (arrives with C).
- No data editing/writing back — read-only analysis.
- Reuse the existing kernel + `OutputView` render path; **no new visualization code**.

*Once you settle §6, I'll write the formal spec to `docs/superpowers/specs/` and move to a writing-plans pass.*

In [2]:
from IPython.display import HTML, display

html = """
<style>
.ds-cmp{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;max-width:780px;border:1px solid #e2e8f0;border-radius:10px;overflow:hidden}
.ds-cmp h2{margin:0;padding:14px 18px;font-size:15px;background:#0f172a;color:#fff}
.ds-cmp input[name=opt]{display:none}
.ds-cmp .tabs{display:flex;border-bottom:1px solid #e2e8f0}
.ds-cmp .tab{flex:1;padding:10px 14px;text-align:center;cursor:pointer;font-size:13px;color:#475569;background:#f8fafc}
#dsA:checked ~ .tabs label[for=dsA],#dsC:checked ~ .tabs label[for=dsC]{background:#fff;color:#0f172a;font-weight:600;box-shadow:inset 0 -2px 0 #22c55e}
.ds-cmp .panel{display:none;padding:18px}
#dsA:checked ~ .panelA{display:block}
#dsC:checked ~ .panelC{display:block}
.flow{font-family:ui-monospace,monospace;font-size:12px;background:#f1f5f9;padding:10px;border-radius:6px;color:#334155;line-height:1.5}
.chips{display:flex;flex-wrap:wrap;gap:6px;margin:12px 0}
.chip{font-size:11px;padding:3px 8px;border-radius:999px}
.yes{background:#dcfce7;color:#166534}.no{background:#fee2e2;color:#991b1b}.mid{background:#fef9c3;color:#854d0e}
.verdict{font-size:13px;padding:10px 12px;border-radius:6px;margin-top:8px}
.vA{background:#fff7ed;color:#9a3412}.vC{background:#ecfdf5;color:#065f46}
#dsReadout{font-size:12px;color:#64748b;padding:8px 18px;border-top:1px solid #e2e8f0}
</style>
<div class="ds-cmp">
  <h2>Representation fork &mdash; how an attached datasource is stored</h2>
  <input type="radio" name="opt" id="dsA"><input type="radio" name="opt" id="dsC" checked>
  <div class="tabs"><label class="tab" for="dsA">A &mdash; setup cell only</label><label class="tab" for="dsC">C &mdash; catalog-backed (recommended)</label></div>
  <div class="panel panelA">
    <div class="flow">sidebar drag-drop &rarr; insert CREATE VIEW cell &rarr; brain reads cell &rarr; brain runs DESCRIBE &rarr; analysis</div>
    <div class="chips"><span class="chip yes">reproducible</span><span class="chip no">brain needs DESCRIBE round-trip</span><span class="chip yes">zero new wire types</span><span class="chip no">no headless catalog</span></div>
    <div class="verdict vA">Smallest MVP. The brain is briefly blind to a GUI attach until it probes. Good if you want minimal surface now.</div>
  </div>
  <div class="panel panelC">
    <div class="flow">sidebar drag-drop &rarr; AttachDatasource cmd &rarr; daemon { inserts setup cell + records schema catalog } &rarr; brain schema-aware instantly &rarr; analysis</div>
    <div class="chips"><span class="chip yes">reproducible</span><span class="chip yes">brain aware, no round-trip</span><span class="chip mid">+1 command, +catalog struct</span><span class="chip yes">headless-ready</span></div>
    <div class="verdict vC">Closes the TUI&harr;GUI seam: a GUI drag-drop becomes instantly legible to the terminal brain. Recommended.</div>
  </div>
  <div id="dsReadout">Toggle the tabs above to compare.</div>
</div>
<script>
(function(){
  var a=document.getElementById('dsA'),c=document.getElementById('dsC'),r=document.getElementById('dsReadout');
  function upd(){r.textContent = c.checked ? 'Selected: C — recommended (catalog + reproducible setup cell).' : 'Selected: A — minimal (setup cell only, brain runs DESCRIBE).';}
  a.addEventListener('change',upd);c.addEventListener('change',upd);upd();
})();
</script>
"""
display(HTML(html))